In [ ]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
# =========================================================
# PARTE 1 - RAG PIPELINE BASE
# =========================================================

!pip -q install sentence-transformers faiss-cpu wikipedia rank_bm25 nltk

import numpy as np
import wikipedia
import faiss
import nltk
import torch

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

nltk.download('punkt')

print("GPU disponible:", torch.cuda.is_available())
!nvidia-smi

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 75.4 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


GPU disponible: True
Wed Mar 11 18:27:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------------

In [ ]:
# =========================================================
# 1. MINI EJEMPLO SIMPLE
# =========================================================

mini_docs = [
    "Artificial intelligence is the simulation of human intelligence in machines.",
    "Machine learning is a subset of artificial intelligence focused on learning from data.",
    "Deep learning is a branch of machine learning based on neural networks with many layers."
]

mini_embedder = SentenceTransformer("all-MiniLM-L6-v2")
mini_embeddings = mini_embedder.encode(mini_docs, convert_to_numpy=True)

dim = mini_embeddings.shape[1]
mini_index = faiss.IndexFlatL2(dim)
mini_index.add(mini_embeddings)

mini_query = "What is machine learning?"
mini_q_emb = mini_embedder.encode([mini_query], convert_to_numpy=True)
D, I = mini_index.search(mini_q_emb, k=2)

mini_results = [mini_docs[i] for i in I[0]]
print("Mini resultados:")
for r in mini_results:
    print("-", r)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Mini resultados:
- Machine learning is a subset of artificial intelligence focused on learning from data.
- Deep learning is a branch of machine learning based on neural networks with many layers.


In [ ]:
# =========================================================
# 2. DATASET EXTERNO REAL (WIKIPEDIA)
# =========================================================

topics = [
    "Artificial intelligence",
    "Machine learning",
    "Deep learning",
    "Neural network",
    "Large language model",
    "Natural language processing",
    "Information retrieval",
    "Question answering",
    "Transformer (deep learning architecture)",
    "Vector database"
]

raw_text = ""

for topic in topics:
    try:
        page = wikipedia.page(topic, auto_suggest=False)
        raw_text += "\n\n" + page.content
        print(f"OK: {topic}")
    except Exception as e:
        print(f"No se pudo cargar {topic}: {e}")

print("\nTotal de caracteres cargados:", len(raw_text))
print(raw_text[:1000])

OK: Artificial intelligence
OK: Machine learning
OK: Deep learning
OK: Neural network
OK: Large language model
OK: Natural language processing
OK: Information retrieval
OK: Question answering
OK: Transformer (deep learning architecture)
OK: Vector database

Total de caracteres cargados: 460627


Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.
High-profile applications of AI include advanced web search engines (e.g., Google Search); recommendation systems (used by YouTube, Amazon, and Netflix); virtual assistants (e.g., Google Assistant, Siri, and Alexa); autonomous vehicles (e.

In [ ]:
# =========================================================
# 3. CHUNKING CON OVERLAP
# =========================================================

def chunk_text(text, chunk_size=250, overlap=50, min_words=40):
    words = text.split()
    chunks = []
    step = chunk_size - overlap

    for i in range(0, len(words), step):
        chunk_words = words[i:i + chunk_size]
        if len(chunk_words) >= min_words:
            chunks.append(" ".join(chunk_words))

    return chunks

chunks = chunk_text(raw_text, chunk_size=250, overlap=50)

print("Número total de chunks:", len(chunks))
print("\nPrimer chunk:\n")
print(chunks[0][:1000])

Número total de chunks: 303

Primer chunk:

Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals. High-profile applications of AI include advanced web search engines (e.g., Google Search); recommendation systems (used by YouTube, Amazon, and Netflix); virtual assistants (e.g., Google Assistant, Siri, and Alexa); autonomous vehicles (e.g., Waymo); generative and creative tools (e.g., language models and AI art); and superhuman play and analysis in strategy games (e.g., chess and Go). However, many AI applications are not perceived as such: "A lot of cutting-edge AI has filtered into g

In [ ]:
# =========================================================
# 4. VALIDAR MINIMO 100 CHUNKS
# =========================================================

if len(chunks) >= 100:
    print(f"✅ Requisito cumplido: {len(chunks)} chunks generados.")
else:
    print(f"⚠️ Solo se generaron {len(chunks)} chunks. Agrega más texto o reduce chunk_size.")

✅ Requisito cumplido: 303 chunks generados.


In [ ]:
# =========================================================
# 5. EMBEDDINGS
# =========================================================

embedder = SentenceTransformer("all-MiniLM-L6-v2")
chunk_embeddings = embedder.encode(chunks, convert_to_numpy=True, show_progress_bar=True)

print("Shape de embeddings:", chunk_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Shape de embeddings: (303, 384)


In [ ]:
# =========================================================
# 6. FAISS HNSW
# =========================================================

dimension = chunk_embeddings.shape[1]

index = faiss.IndexHNSWFlat(dimension, 32)
index.hnsw.efConstruction = 40
index.hnsw.efSearch = 32
index.add(chunk_embeddings)

print("Vectores indexados:", index.ntotal)

Vectores indexados: 303


In [ ]:
# =========================================================
# 7. BM25
# =========================================================

tokenized_chunks = [chunk.split() for chunk in chunks]
bm25 = BM25Okapi(tokenized_chunks)

print("BM25 listo.")

BM25 listo.


In [ ]:
# =========================================================
# 8. FUNCIONES DE RETRIEVAL
# =========================================================

def retrieve_faiss(query, top_k=5):
    q_emb = embedder.encode([query], convert_to_numpy=True)
    distances, indices = index.search(q_emb, top_k)
    return [(int(i), chunks[int(i)], float(distances[0][j])) for j, i in enumerate(indices[0])]

def retrieve_bm25(query, top_k=5):
    scores = bm25.get_scores(query.split())
    best_idx = np.argsort(scores)[::-1][:top_k]
    return [(int(i), chunks[int(i)], float(scores[i])) for i in best_idx]

def hybrid_retrieve(query, top_k_faiss=5, top_k_bm25=5, final_k=8):
    faiss_results = retrieve_faiss(query, top_k=top_k_faiss)
    bm25_results = retrieve_bm25(query, top_k=top_k_bm25)

    merged = {}

    for idx, text, score in faiss_results:
        merged[idx] = {
            "chunk": text,
            "faiss_score": score,
            "bm25_score": 0.0
        }

    for idx, text, score in bm25_results:
        if idx not in merged:
            merged[idx] = {
                "chunk": text,
                "faiss_score": 0.0,
                "bm25_score": score
            }
        else:
            merged[idx]["bm25_score"] = score

    faiss_vals = np.array([v["faiss_score"] for v in merged.values()], dtype=float)
    bm25_vals = np.array([v["bm25_score"] for v in merged.values()], dtype=float)

    faiss_norm = (faiss_vals.max() - faiss_vals) / (faiss_vals.max() - faiss_vals.min() + 1e-9)
    bm25_norm = (bm25_vals - bm25_vals.min()) / (bm25_vals.max() - bm25_vals.min() + 1e-9)

    keys = list(merged.keys())
    final = []

    for i, idx in enumerate(keys):
        combined = 0.5 * faiss_norm[i] + 0.5 * bm25_norm[i]
        final.append((idx, merged[idx]["chunk"], float(combined)))

    final = sorted(final, key=lambda x: x[2], reverse=True)[:final_k]
    return final

In [ ]:
# =========================================================
# 9. RERANKER CROSS-ENCODER
# =========================================================

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank_results(query, retrieved_items, top_k=5):
    pairs = [[query, item[1]] for item in retrieved_items]
    scores = reranker.predict(pairs)

    ranked = []
    for item, score in zip(retrieved_items, scores):
        ranked.append((item[0], item[1], float(score)))

    ranked = sorted(ranked, key=lambda x: x[2], reverse=True)[:top_k]
    return ranked

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [ ]:
# =========================================================
# 10. PRUEBA DEL PIPELINE BASE
# =========================================================

query = "What is the difference between artificial intelligence and machine learning?"

retrieved = hybrid_retrieve(query, final_k=8)
reranked = rerank_results(query, retrieved, top_k=5)

print("=== RECUPERACIÓN HÍBRIDA ===\n")
for idx, chunk, score in retrieved:
    print(f"chunk_{idx} | hybrid_score={score:.4f}")
    print(chunk[:300], "\n")

print("\n=== RERANKING FINAL ===\n")
for idx, chunk, score in reranked:
    print(f"chunk_{idx} | rerank_score={score:.4f}")
    print(chunk[:400], "\n")

=== RECUPERACIÓN HÍBRIDA ===

chunk_39 | hybrid_score=1.0000
to redundancies. In April 2023, it was reported that 70% of the jobs for Chinese video game illustrators had been eliminated by generative artificial intelligence. Early-career workers showed decreasing employment rates in some AI-exposed occupations. Unlike previous waves of automation, many middle 

chunk_84 | hybrid_score=0.9331
for instances that seem to fit the least to the remainder of the data set. Supervised anomaly detection techniques require a data set that has been labelled as "normal" and "abnormal" and involves training a classifier (the key difference from many other statistical classification problems is the in 

chunk_146 | hybrid_score=0.9017
the lack of theory surrounding some methods. Learning in the most common deep architectures is implemented using well-understood gradient descent. However, the theory surrounding other algorithms, such as contrastive divergence is less clear. (e.g., Does it converge? If

In [ ]:
# =========================================================
# PARTE 2 - GENERACION Y EVALUACION
# =========================================================

!pip -q install transformers rouge-score accelerate scikit-learn

  Preparing metadata (setup.py) ... done


In [ ]:
# =========================================================
# 11. IMPORTS EXTRA
# =========================================================

import re
from rouge_score import rouge_scorer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
# =========================================================
# 12. CARGAR QWEN
# =========================================================

model_name = "Qwen/Qwen2-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForCausalLM.from_pretrained(model_name)

print("Qwen cargado correctamente.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen cargado correctamente.


In [ ]:
# =========================================================
# 13. CONSTRUIR PROMPT
# =========================================================

def build_prompt(query, ranked_chunks):
    context_blocks = []

    for idx, chunk, score in ranked_chunks:
        context_blocks.append(f"[chunk_{idx}]\n{chunk}")

    context = "\n\n".join(context_blocks)

    prompt = f"""
You are a grounded RAG assistant.
Answer the question using ONLY the provided context.
If the answer is not supported by the context, say that the information is not sufficiently grounded.

Context:
{context}

Question:
{query}

Answer:
"""

    return prompt, context

In [ ]:
# =========================================================
# 14. GENERAR RESPUESTA RAG
# =========================================================

def generate_answer(query, top_k_hybrid=8, top_k_rerank=5, max_new_tokens=120):
    retrieved = hybrid_retrieve(query, final_k=top_k_hybrid)
    reranked = rerank_results(query, retrieved, top_k=top_k_rerank)

    prompt, context = build_prompt(query, reranked)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    outputs = llm.generate(**inputs, max_new_tokens=max_new_tokens)

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "Answer:" in decoded:
        answer = decoded.split("Answer:")[-1].strip()
    else:
        answer = decoded.strip()

    return {
        "retrieved": retrieved,
        "reranked": reranked,
        "context": context,
        "prompt": prompt,
        "answer": answer
    }

In [ ]:
# =========================================================
# 15. PROBAR GENERACION
# =========================================================

query = "What is the difference between artificial intelligence and machine learning?"
result = generate_answer(query)

print("=== RESPUESTA ===\n")
print(result["answer"])

print("\n=== CHUNKS RERANKEADOS ===\n")
for idx, chunk, score in result["reranked"]:
    print(f"chunk_{idx} | rerank_score={score:.4f}")
    print(chunk[:350], "\n")

=== RESPUESTA ===

The main difference between artificial intelligence and machine learning is that artificial intelligence aims to achieve artificial intelligence through the development of specific methods and tools, whereas machine learning is a subset of AI that focuses on the use of machine learning algorithms and methods to recognize patterns in data and generate predictions. Artificial intelligence refers to the ability of computational systems to perform tasks typically associated with human intelligence, while machine learning is a field of research in computer science that deals with the development and study of algorithms that enable machines to learn and solve problems from data.

=== CHUNKS RERANKEADOS ===

chunk_0 | rerank_score=3.1727
Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research 

In [ ]:
# =========================================================
# 16. CITATION PER SENTENCE
# =========================================================

def split_sentences(text):
    sents = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s for s in sents if s.strip()]

def cite_per_sentence(answer, reranked_chunks):
    sentences = split_sentences(answer)

    if not sentences:
        return []

    chunk_ids = [item[0] for item in reranked_chunks]
    chunk_texts = [item[1] for item in reranked_chunks]

    sent_emb = embedder.encode(sentences, convert_to_numpy=True)
    chunk_emb = embedder.encode(chunk_texts, convert_to_numpy=True)

    cited = []

    for sent, emb in zip(sentences, sent_emb):
        sims = cosine_similarity([emb], chunk_emb)[0]
        best_idx = int(np.argmax(sims))

        cited.append({
            "sentence": sent,
            "citation": f"chunk_{chunk_ids[best_idx]}",
            "similarity": float(sims[best_idx])
        })

    return cited

In [ ]:
# =========================================================
# 17. MOSTRAR CITAS
# =========================================================

citations = cite_per_sentence(result["answer"], result["reranked"])

for item in citations:
    print(f'{item["sentence"]} [{item["citation"]}] sim={item["similarity"]:.4f}')

The main difference between artificial intelligence and machine learning is that artificial intelligence aims to achieve artificial intelligence through the development of specific methods and tools, whereas machine learning is a subset of AI that focuses on the use of machine learning algorithms and methods to recognize patterns in data and generate predictions. [chunk_70] sim=0.6507
Artificial intelligence refers to the ability of computational systems to perform tasks typically associated with human intelligence, while machine learning is a field of research in computer science that deals with the development and study of algorithms that enable machines to learn and solve problems from data. [chunk_0] sim=0.6693


In [ ]:
# =========================================================
# 18. HALLUCINATION GUARD / GROUNDING SCORE
# =========================================================

def grounding_score(answer, context):
    ans_emb = embedder.encode([answer], convert_to_numpy=True)
    ctx_emb = embedder.encode([context], convert_to_numpy=True)

    score = cosine_similarity(ans_emb, ctx_emb)[0][0]
    return float(score)

g_score = grounding_score(result["answer"], result["context"])

print("Grounding score:", round(g_score, 4))

if g_score < 0.30:
    print("⚠️ Posible alucinación")
elif g_score < 0.70:
    print("🟡 Respuesta parcialmente sustentada")
else:
    print("✅ Respuesta bien fundamentada")

Grounding score: 0.6138
🟡 Respuesta parcialmente sustentada


In [ ]:
# =========================================================
# 19. EVALUACION AUTOMATICA CON ROUGE
# =========================================================

reference_answer = (
    "Artificial intelligence is a broad field focused on creating systems capable of tasks "
    "that normally require human intelligence. Machine learning is a subset of artificial "
    "intelligence that learns patterns from data."
)

scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
rouge_scores = scorer.score(reference_answer, result["answer"])

print("ROUGE-1:", rouge_scores["rouge1"])
print("ROUGE-L:", rouge_scores["rougeL"])

ROUGE-1: Score(precision=0.26, recall=0.8387096774193549, fmeasure=0.39694656488549623)
ROUGE-L: Score(precision=0.19, recall=0.6129032258064516, fmeasure=0.2900763358778626)


In [ ]:
# =========================================================
# 20. SALIDA FINAL PARA LA DEMO
# =========================================================

print("PREGUNTA:\n", query)
print("\nRESPUESTA:\n", result["answer"])
print("\nGROUNDING SCORE:\n", round(g_score, 4))

print("\nCITAS POR ORACIÓN:")
for item in citations:
    print(f'- {item["sentence"]} [{item["citation"]}]')

print("\nCHUNKS USADOS:")
for idx, chunk, score in result["reranked"]:
    print(f"\n--- chunk_{idx} | rerank_score={score:.4f} ---")
    print(chunk[:700])

PREGUNTA:
 What is the difference between artificial intelligence and machine learning?

RESPUESTA:
 The main difference between artificial intelligence and machine learning is that artificial intelligence aims to achieve artificial intelligence through the development of specific methods and tools, whereas machine learning is a subset of AI that focuses on the use of machine learning algorithms and methods to recognize patterns in data and generate predictions. Artificial intelligence refers to the ability of computational systems to perform tasks typically associated with human intelligence, while machine learning is a field of research in computer science that deals with the development and study of algorithms that enable machines to learn and solve problems from data.

GROUNDING SCORE:
 0.6138

CITAS POR ORACIÓN:
- The main difference between artificial intelligence and machine learning is that artificial intelligence aims to achieve artificial intelligence through the development 

In [ ]:
query = "What is machine learning?"
result = generate_answer(query)

print(result["answer"])

Machine learning is a branch of artificial intelligence concerned with enabling machines to mimic the human brain in performing tasks traditionally performed by humans. It encompasses a wide range of algorithms that enable machines to process data, recognize patterns, and generate new information. Some examples of machine learning include clustering, classification, regression, and natural language processing. The term "machine learning" is derived from the name of the person who first coined it, Arthur Samuel.


In [ ]:
citations = cite_per_sentence(result["answer"], result["reranked"])
g_score = grounding_score(result["answer"], result["context"])
print(g_score)

0.7631633281707764
